# Titanic Survival Prediction

**A complete beginner-friendly classification workflow**

This notebook explores the Titanic passenger dataset, prepares selected features, trains and evaluates a Random Forest classifier, and exports the complete Scikit-learn pipeline as `titanic_model.pkl` for the Flask application.

**Target:** `Survived` (`0` = did not survive, `1` = survived).

**Important:** This is an educational model. Its predictions and probabilities are estimates learned from historical data, not historical certainty or causal explanations.

## Step 1 — Import libraries

We use Pandas and NumPy for data work, Matplotlib and Seaborn for visualisation, and Scikit-learn for preprocessing, training, and evaluation.

In [ ]:
from pathlib import Path
import json
import pickle
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

print("Libraries imported successfully.")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

## Step 2 — Load and inspect the dataset

Keep `Titanic-Dataset.csv` in the same directory as this notebook. We inspect the first rows, dimensions, column names, data types, and summary statistics before modelling.

In [ ]:
DATA_PATH = Path("Titanic-Dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Titanic-Dataset.csv was not found. Put the CSV beside this notebook."
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head())

In [ ]:
df.info()

In [ ]:
display(df.describe(include="all").T)

## Step 3 — Check data quality and missing values

Missing values are expected in some passenger columns. We will not fill them manually here; the model pipeline in Step 7 will learn imputation values from the training split only. This prevents test data from influencing preprocessing.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

display(missing_summary)

In [ ]:
plt.figure(figsize=(11, 4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap="mako")
plt.title("Missing values across Titanic records")
plt.xlabel("Dataset columns")
plt.tight_layout()
plt.show()

## Step 4 — Exploratory data analysis (EDA)

These plots show how survival is distributed in the dataset and how it varies across selected passenger characteristics. Associations in these charts do not prove cause and effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Survived", ax=axes[0], hue="Survived", legend=False, palette="Set2")
axes[0].set_title("Passenger survival count")
axes[0].set_xlabel("Survived (0 = No, 1 = Yes)")
axes[0].set_ylabel("Passengers")

survival_rate = df["Survived"].mean() * 100
axes[1].pie(
    [100 - survival_rate, survival_rate],
    labels=["Did not survive", "Survived"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#8093ad", "#40a987"],
)
axes[1].set_title("Share of passengers by outcome")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(data=df, x="Sex", hue="Survived", ax=axes[0], palette="Set2")
axes[0].set_title("Survival by sex")

sns.countplot(data=df, x="Pclass", hue="Survived", ax=axes[1], palette="Set2")
axes[1].set_title("Survival by passenger class")

sns.histplot(data=df, x="Age", hue="Survived", bins=25, kde=True, ax=axes[2], element="step")
axes[2].set_title("Age distribution by outcome")

plt.tight_layout()
plt.show()

In [ ]:
embarked_plot = df.dropna(subset=["Embarked"])
plt.figure(figsize=(7, 4))
sns.countplot(data=embarked_plot, x="Embarked", hue="Survived", palette="Set2")
plt.title("Survival by port of embarkation")
plt.xlabel("Embarked (C = Cherbourg, Q = Queenstown, S = Southampton)")
plt.ylabel("Passengers")
plt.tight_layout()
plt.show()

In [ ]:
print("Survival rate by sex:")
display(df.groupby("Sex", dropna=False)["Survived"].agg(["count", "mean"]).rename(columns={"mean": "survival_rate"}))

print("Survival rate by passenger class:")
display(df.groupby("Pclass")["Survived"].agg(["count", "mean"]).rename(columns={"mean": "survival_rate"}))

## Step 5 — Select features and target

We use seven features that are also collected by the Flask form. `Pclass` is treated as categorical for the model. We exclude identifiers and free-text columns such as passenger name and ticket number.

In [ ]:
FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
TARGET = "Survived"

required_columns = FEATURES + [TARGET]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

model_data = df[required_columns].copy()
model_data = model_data.dropna(subset=[TARGET])

X = model_data[FEATURES]
y = model_data[TARGET].astype(int)

print("Feature matrix:", X.shape)
print("Target vector:", y.shape)
print("Target classes:")
display(y.value_counts().sort_index().rename(index={0: "Did not survive", 1: "Survived"}))

## Step 6 — Split training and testing data

We reserve 20% of records for final evaluation. `stratify=y` keeps the survival-class proportions similar in both splits. The random state makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target distribution:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion"))
print("Testing target distribution:")
display(y_test.value_counts(normalize=True).sort_index().rename("proportion"))

## Step 7 — Build preprocessing and train the model

Numerical columns use median imputation. Categorical columns use most-frequent imputation followed by one-hot encoding. Both preprocessing and the Random Forest classifier are kept in one pipeline, so Flask receives the same transformations as the training data.

In [ ]:
NUMERIC_FEATURES = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_FEATURES = ["Pclass", "Sex", "Embarked"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, NUMERIC_FEATURES),
    ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

model.fit(X_train, y_train)
print("Model training completed.")

## Step 8 — Evaluate model performance

We evaluate against the held-out test set using accuracy, precision, recall, F1-score, a classification report, and a confusion matrix. Accuracy alone does not describe all types of classification errors.

In [ ]:
y_pred = model.predict(X_test)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 score": f1_score(y_test, y_pred, zero_division=0),
}

metrics_df = pd.DataFrame({"Metric": list(metrics.keys()), "Score": list(metrics.values())})
display(metrics_df.style.format({"Score": "{:.2%}"}))

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Did not survive", "Survived"],
    zero_division=0,
))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Did not survive", "Survived"],
    yticklabels=["Did not survive", "Survived"],
)
plt.title("Confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.tight_layout()
plt.show()

## Step 9 — Inspect feature importance

Random Forest provides impurity-based feature-importance scores for the transformed features. These can help with model inspection, but they should not be interpreted as proof that a feature causes survival. Importance can also be biased by feature characteristics.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances,
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df.head(12), x="Importance", y="Feature", color="#536dfe")
plt.title("Top transformed feature importances")
plt.tight_layout()
plt.show()

## Step 10 — Export and test the Pickle model

This step saves the **entire fitted pipeline**, not just the classifier. Flask loads this same file and sends it a DataFrame with the expected feature names. Only load Pickle files that you created or otherwise trust. For reliable loading, keep the same Python and Scikit-learn versions between training and deployment.

In [ ]:
MODEL_PATH = Path("titanic_model.pkl")
METADATA_PATH = Path("model_metadata.json")

with MODEL_PATH.open("wb") as file:
    pickle.dump(model, file, protocol=pickle.HIGHEST_PROTOCOL)

metadata = {
    "model_name": "RandomForestClassifier pipeline",
    "features": FEATURES,
    "target": TARGET,
    "python_version": platform.python_version(),
    "scikit_learn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "test_metrics": {name: float(value) for name, value in metrics.items()},
}

METADATA_PATH.write_text(json.dumps(metadata, indent=4), encoding="utf-8")

print(f"Saved model to: {MODEL_PATH.resolve()}")
print(f"Saved metadata to: {METADATA_PATH.resolve()}")

In [ ]:
# Reload the saved file and test a sample passenger.
with MODEL_PATH.open("rb") as file:
    loaded_model = pickle.load(file)

sample_passenger = pd.DataFrame([{
    "Pclass": 1,
    "Sex": "female",
    "Age": 28.0,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 80.0,
    "Embarked": "C",
}], columns=FEATURES)

sample_prediction = int(loaded_model.predict(sample_passenger)[0])
sample_probability = float(loaded_model.predict_proba(sample_passenger)[0, 1])

print("Prediction:", "Survived" if sample_prediction == 1 else "Did not survive")
print(f"Estimated survival probability: {sample_probability:.1%}")
print("Export and reload test passed.")